In [34]:
import os
import pandas as pd

from finrl.meta.env_stock_trading.env_stocktrading import StockTradingEnv
from finrl.agents.stablebaselines3.models import DRLAgent
from stable_baselines3.common.logger import configure
from finrl import config_tickers
from finrl.main import check_and_make_directories
from finrl.config import INDICATORS, TRAINED_MODEL_DIR, RESULTS_DIR

check_and_make_directories([TRAINED_MODEL_DIR])
SEED = [0, 1, 2, 3, 4]

In [35]:
"""
Trade-Republic-Kosten für das FinRL StockTradingEnv (Stand: September 2026).

Berücksichtigt nur:
  - Ordergebühr: 1 € (Bestpreis) bzw. 2 € (Direktpreis) fix pro Order
  - Transaktionskosten: geschätzter Spread in Prozent

from finrl.meta.env_stock_trading.env_stocktrading import StockTradingEnv


class TradeRepublicStockTradingEnv(StockTradingEnv):
    def __init__(self, df, *, order_fee=1.0, spread_pct=0.0005, **kwargs):
        self.order_fee = order_fee      # 1.0 = Bestpreis, 2.0 = Direktpreis
        self.spread_pct = spread_pct    # z. B. 0.0005 = 0,05 %

        # Prozentuale Kosten des Basis-Envs abschalten
        n = kwargs["stock_dim"]
        kwargs["buy_cost_pct"] = [0.0] * n
        kwargs["sell_cost_pct"] = [0.0] * n
        super().__init__(df=df, **kwargs)

    def _turbulent(self):
        return (self.turbulence_threshold is not None
                and self.turbulence >= self.turbulence_threshold)

    def _sell_stock(self, index, action):
        price = self.state[1 + index]
        held = self.state[1 + self.stock_dim + index]
        if price <= 0 or held <= 0:
            return 0

        # Bei Turbulenz alles verkaufen (wie im Basis-Env)
        shares = int(held if self._turbulent() else min(abs(action), held))
        proceeds = price * shares * (1 - self.spread_pct)
        if shares <= 0 or proceeds <= self.order_fee:
            return 0

        self.state[0] += proceeds - self.order_fee
        self.state[1 + self.stock_dim + index] -= shares
        self.cost += price * shares * self.spread_pct + self.order_fee
        self.trades += 1
        return shares

    def _buy_stock(self, index, action):
        price = self.state[1 + index]
        if price <= 0 or self._turbulent():
            return 0

        ask = price * (1 + self.spread_pct)
        affordable = int((self.state[0] - self.order_fee) // ask)
        shares = int(min(affordable, action))
        if shares <= 0:
            return 0

        self.state[0] -= ask * shares + self.order_fee
        self.state[1 + self.stock_dim + index] += shares
        self.cost += price * shares * self.spread_pct + self.order_fee
        self.trades += 1
        return shares  """

'\nTrade-Republic-Kosten für das FinRL StockTradingEnv (Stand: September 2026).\n\nBerücksichtigt nur:\n  - Ordergebühr: 1 € (Bestpreis) bzw. 2 € (Direktpreis) fix pro Order\n  - Transaktionskosten: geschätzter Spread in Prozent\n\nfrom finrl.meta.env_stock_trading.env_stocktrading import StockTradingEnv\n\n\nclass TradeRepublicStockTradingEnv(StockTradingEnv):\n    def __init__(self, df, *, order_fee=1.0, spread_pct=0.0005, **kwargs):\n        self.order_fee = order_fee      # 1.0 = Bestpreis, 2.0 = Direktpreis\n        self.spread_pct = spread_pct    # z. B. 0.0005 = 0,05 %\n\n        # Prozentuale Kosten des Basis-Envs abschalten\n        n = kwargs["stock_dim"]\n        kwargs["buy_cost_pct"] = [0.0] * n\n        kwargs["sell_cost_pct"] = [0.0] * n\n        super().__init__(df=df, **kwargs)\n\n    def _turbulent(self):\n        return (self.turbulence_threshold is not None\n                and self.turbulence >= self.turbulence_threshold)\n\n    def _sell_stock(self, index, action)

In [36]:
from finrl.meta.env_stock_trading.env_stocktrading import StockTradingEnv

class TRStockTradingEnv(StockTradingEnv):
    """StockTradingEnv mit Trade-Republic-Fixgebuehr von 1 EUR je Ausfuehrung.

    Die geerbten buy_cost_pct / sell_cost_pct bleiben als Spread aktiv.
    Quelle der Kondition: Preis- und Leistungsverzeichnis Trade Republic.
    """

    def __init__(self, *args, fixed_fee=1.0, **kwargs):
        super().__init__(*args, **kwargs)
        self.fixed_fee = fixed_fee
        self.fee_total = 0.0

    def _charge_fee(self, shares):
        if shares > 0:
            self.state[0] -= self.fixed_fee   # Cash belasten
            self.cost += self.fixed_fee       # in FinRLs Kostenzaehler
            self.fee_total += self.fixed_fee  # eigener Zaehler fuer die Auswertung

    def _buy_stock(self, index, action):
        shares = super()._buy_stock(index, action)
        self._charge_fee(shares)
        return shares

    def _sell_stock(self, index, action):
        shares = super()._sell_stock(index, action)
        self._charge_fee(shares)
        return shares

    def reset(self, *, seed=None, options=None):
        out = super().reset(seed=seed, options=options)
        self.fee_total = 0.0
        return out

In [37]:
#Load Data from 01_data_ops_notebook
train = pd.read_csv('../../data/train.csv')
date = train.columns[0]
train.index = pd.factorize(train[date])[0]
train.index.names = ['']
train.head()


,date,tic,close,high,low,open,volume,day,macd,boll_ub,boll_lb,rsi_30,cci_30,dx_30,close_30_sma,close_60_sma,vix,turbulence
,,,,,,,,,,,,,,,,,,
0,2009-01-02,ADS.DE,22.034235,22.224390,20.996305,21.075537,580197.0,4.0,0.0,22.140397,21.983533,100.0,66.666667,100.0,22.034235,22.034235,39.189999,0.0
0,2009-01-02,ALV.DE,32.786087,33.036815,31.957402,32.726592,1424010.0,4.0,0.0,22.140397,21.983533,100.0,66.666667,100.0,32.786087,32.786087,39.189999,0.0
0,2009-01-02,BAS.DE,11.868842,11.953924,11.643377,11.868842,3752004.0,4.0,0.0,22.140397,21.983533,100.0,66.666667,100.0,11.868842,11.868842,39.189999,0.0
0,2009-01-02,BAYN.DE,26.211126,26.253964,25.678703,25.996931,3088175.0,4.0,0.0,22.140397,21.983533,100.0,66.666667,100.0,26.211126,26.211126,39.189999,0.0
0,2009-01-02,BEI.DE,34.603619,35.324017,34.505384,34.726412,356642.0,4.0,0.0,22.140397,21.983533,100.0,66.666667,100.0,34.603619,34.603619,39.189999,0.0


In [38]:
#Setup Environment
stock_dimensions = len(train.tic.unique())
state_space = 1 + 2 * stock_dimensions + len(INDICATORS)*stock_dimensions
print(f"Stock Dimensions:  {stock_dimensions}, State Space: {state_space}")

Stock Dimensions:  31, State Space: 311


In [39]:
buy_cost_list = sell_cost_list = [0.0005] * stock_dimensions
num_stock_shares = [0] * stock_dimensions

env_tweeks = {
    "hmax": 50,
    "initial_amount": 100000,
    "num_stock_shares": num_stock_shares,
    "buy_cost_pct": buy_cost_list,
    "sell_cost_pct": sell_cost_list,
    "state_space": state_space,
    "stock_dim": stock_dimensions,
    "tech_indicator_list": INDICATORS,
    "action_space": stock_dimensions,
    "reward_scaling": 1e-4
}
#Setup of the environment with given attributes and data set
e_training_gym = TRStockTradingEnv(df = train, fixed_fee=1.0, **env_tweeks)


In [40]:
#Environment for Training

env_train, _ = e_training_gym.get_sb_env()
print(type(env_train))

<class 'stable_baselines3.common.vec_env.dummy_vec_env.DummyVecEnv'>


In [41]:
#Setup Agent

agent = DRLAgent(env = env_train)

#Set if_using_MODEL = true if u want to use other algorithms
if_using_a2c = True
if_using_ddpg = False
if_using_ppo = True
if_using_td3 = False
if_using_sac = True

In [42]:
from stable_baselines3.common.callbacks import BaseCallback
class PortfolioCallback(BaseCallback):
    """Loggt Depotwert, Kosten und Orderzahl aus dem Environment."""

    def _on_step(self) -> bool:
        env = self.training_env                       # DummyVecEnv aus get_sb_env()
        self.logger.record("portfolio/asset_value", 
        env.get_attr("asset_memory")[0][-1])
        self.logger.record("portfolio/cost_cum",    env.get_attr("cost")[0])
        self.logger.record("portfolio/trades_cum",  env.get_attr("trades")[0])
        self.logger.record("portfolio/fee_total",   env.get_attr("fee_total")[0])
        return True

In [43]:
from collections.abc import Set
from turtle import setup


if if_using_a2c:
    for seed in SEED:
        #Using model A2C
        model_a2c =agent.get_model("a2c", seed=seed)
        #setup new logger
        tmp_path = RESULTS_DIR + '/a2c'
        new_logger_a2c = configure(tmp_path, ["stdout", "csv", "tensorboard"])
        #Set new logger
        model_a2c.set_logger(new_logger_a2c)

if if_using_ddpg:
    for seed in SEED:
        #Using model DDPG
        model_ddpg =agent.get_model("ddpg", seed=seed)
        #setup new logger
        tmp_path = RESULTS_DIR + '/ddpg'
        new_logger_ddpg = configure(tmp_path, ["stdout", "csv", "tensorboard"])
        #Set new logger
        model_ddpg.set_logger(new_logger_ddpg)

if if_using_ppo:
    for seed in SEED:
        #Using model PPO
        model_ppo =agent.get_model("ppo", seed=seed)
        #setup new logger
        tmp_path = RESULTS_DIR + '/ppo'
        new_logger_ppo = configure(tmp_path, ["stdout", "csv", "tensorboard"])
        #Set new logger
        model_ppo.set_logger(new_logger_ppo)


if if_using_td3:
    for seed in SEED:
        #Using model TD3
        model_td3 =agent.get_model("td3", seed=seed)
        #setup new logger
        tmp_path = RESULTS_DIR + '/td3'
        new_logger_td3 = configure(tmp_path, ["stdout", "csv", "tensorboard"])
        #Set new logger
        model_td3.set_logger(new_logger_td3)

if if_using_sac:
    for seed in SEED:
        #Using model SAC
        model_sac =agent.get_model("sac", seed=seed)
        #setup new logger
        tmp_path = RESULTS_DIR + '/sac'
        new_logger_sac = configure(tmp_path, ["stdout", "csv", "tensorboard"])
        #Set new logger
        model_sac.set_logger(new_logger_sac)

{'n_steps': 5, 'ent_coef': 0.01, 'learning_rate': 0.0007}
Using cpu device
Logging to results/a2c
{'n_steps': 5, 'ent_coef': 0.01, 'learning_rate': 0.0007}
Using cpu device
Logging to results/a2c
{'n_steps': 5, 'ent_coef': 0.01, 'learning_rate': 0.0007}
Using cpu device
Logging to results/a2c
{'n_steps': 5, 'ent_coef': 0.01, 'learning_rate': 0.0007}
Using cpu device
Logging to results/a2c


{'n_steps': 5, 'ent_coef': 0.01, 'learning_rate': 0.0007}
Using cpu device
Logging to results/a2c
{'n_steps': 2048, 'ent_coef': 0.01, 'learning_rate': 0.00025, 'batch_size': 64}
Using cpu device
Logging to results/ppo
{'n_steps': 2048, 'ent_coef': 0.01, 'learning_rate': 0.00025, 'batch_size': 64}
Using cpu device
Logging to results/ppo
{'n_steps': 2048, 'ent_coef': 0.01, 'learning_rate': 0.00025, 'batch_size': 64}
Using cpu device
Logging to results/ppo
{'n_steps': 2048, 'ent_coef': 0.01, 'learning_rate': 0.00025, 'batch_size': 64}
Using cpu device
Logging to results/ppo
{'n_steps': 2048, 'ent_coef': 0.01, 'learning_rate': 0.00025, 'batch_size': 64}
Using cpu device
Logging to results/ppo
{'batch_size': 64, 'buffer_size': 100000, 'learning_rate': 0.0001, 'learning_starts': 100, 'ent_coef': 'auto_0.1'}
Using cpu device
Logging to results/sac
{'batch_size': 64, 'buffer_size': 100000, 'learning_rate': 0.0001, 'learning_starts': 100, 'ent_coef': 'auto_0.1'}
Using cpu device
Logging to resu

In [ ]:
#Use loggs to train data. Add aditional blocks for other models if needed
if if_using_a2c:
    trained_a2c = agent.train_model(
        model = model_a2c,
        tb_log_name = 'a2c',
        total_timesteps = 200_000, #Increase for more learning
        callbacks = [PortfolioCallback()]
    )
if if_using_ddpg:
    trained_ddpg = agent.train_model(
        model = model_ddpg,
        tb_log_name = 'ddpg',
        total_timesteps = 200_000, #Increase for more learning
        callbacks = [PortfolioCallback()]
    )
if if_using_ppo:
    trained_ppo = agent.train_model(
        model = model_ppo,
        tb_log_name = 'ppo',
        total_timesteps = 200_000, #Increase for more learning
        callbacks = [PortfolioCallback()]
    )
if if_using_sac:
    trained_sac = agent.train_model(
        model = model_sac,
        tb_log_name = 'sac',
        total_timesteps = 200_000, #Increase for more learning
        callbacks = [PortfolioCallback()]
    )

----------------------------------------
| portfolio/            |              |
|    asset_value        | 1.78e+05     |
|    cost_cum           | 4.42e+03     |
|    fee_total          | 3.31e+03     |
|    trades_cum         | 8811         |
| time/                 |              |
|    fps                | 109          |
|    iterations         | 100          |
|    time_elapsed       | 4            |
|    total_timesteps    | 500          |
| train/                |              |
|    entropy_loss       | -93.8        |
|    explained_variance | 0            |
|    learning_rate      | 0.0007       |
|    n_updates          | 40099        |
|    policy_loss        | -10          |
|    reward             | -0.101926886 |
|    reward_max         | 0.2102013    |
|    reward_mean        | -0.053484686 |
|    reward_min         | -0.2456661   |
|    std                | 5.02         |
|    value_loss         | 0.0181       |
----------------------------------------
----------------

In [ ]:
if if_using_a2c:
    trained_a2c.save(TRAINED_MODEL_DIR + "/agent_a2c")if if_using_a2c else None
if if_using_ddpg:
    trained_ddpg.save(TRAINED_MODEL_DIR + "/agent_ddpg")if if_using_ddpg else None
if if_using_ppo:
    trained_ppo.save(TRAINED_MODEL_DIR + "/agent_ppo")if if_using_ppo else None
if if_using_sac:
    trained_sac.save(TRAINED_MODEL_DIR + "/agent_sac")if if_using_sac else None


In [ ]:
# Portfolio am Ende des Trainingszeitraums auswerten
if if_using_a2c:
    portfolio_model = trained_a2c
elif if_using_ddpg:
    portfolio_model = trained_ddpg
elif if_using_ppo:
    portfolio_model = trained_ppo
elif if_using_sac:
    portfolio_model = trained_sac
else:
    raise ValueError("Aktiviere mindestens ein trainiertes Modell.")

portfolio_env = TRStockTradingEnv(df=train, **env_tweeks)
reset_result = portfolio_env.reset()
obs = reset_result[0] if isinstance(reset_result, tuple) else reset_result

for _ in range(len(train.index.unique())):
    action, _ = portfolio_model.predict(obs, deterministic=True)
    obs, reward, terminated, truncated, info = portfolio_env.step(action)
    if terminated or truncated:
        break

stock_count = portfolio_env.stock_dim
state = portfolio_env.state
cash = float(state[0])
prices = state[1:1 + stock_count]
shares = state[1 + stock_count:1 + 2 * stock_count]

ticker_order = train.tic.drop_duplicates().tolist()
portfolio = pd.DataFrame({
    "Aktie": ticker_order,
    "Stueck": shares,
    "Kurs_EUR": prices,
})
portfolio["Wert_EUR"] = portfolio["Stueck"] * portfolio["Kurs_EUR"]
total_value = cash + portfolio["Wert_EUR"].sum()
portfolio["Anteil_Prozent"] = portfolio["Wert_EUR"] / total_value * 100
portfolio = portfolio.loc[portfolio["Stueck"] > 0].sort_values("Wert_EUR", ascending=False)

print(f"Barbestand: {cash:.2f} EUR | Gesamtportfoliowert: {total_value:.2f} EUR")
if portfolio.empty:
    print("Am Ende des Zeitraums wurden keine Aktien gehalten.")
else:
    display(portfolio.reset_index(drop=True).round({"Kurs_EUR": 2, "Wert_EUR": 2, "Anteil_Prozent": 2}))